# Final Submission(Week 9)
## Team 12

### Imports

In [17]:
import numpy as np
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

### Load & Combine ALL 5 Datasets

In [6]:
files = [
    "processed_dataset1.csv",
    "processed_dataset2.csv",
    "processed_dataset3.csv",
    "processed_dataset4.csv",
    "processed_dataset5.csv"
]

dfs = [pd.read_csv(f) for f in files]
df = pd.concat(dfs, ignore_index=True)

print("Combined dataset shape:", df.shape)

Combined dataset shape: (383119, 9)


### Define Inputs & Labels

In [7]:
X = df["clean_comment"].astype(str)

y = df[['toxic','severe_toxic','obscene','threat','insult','identity_hate']]

### Tokenization

In [8]:
max_words = 50000
max_len = 150

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(X)

X_seq = tokenizer.texts_to_sequences(X)
X_pad = pad_sequences(X_seq, maxlen=max_len)

### Train-Test Split

In [9]:
X_train, X_val, y_train, y_val = train_test_split(
    X_pad, y, test_size=0.2, random_state=42
)

### Load GloVe

In [11]:
embedding_dim = 100
embeddings_index = {}

with open("glove.6B.100d.txt", encoding="utf8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vectors = np.asarray(values[1:], dtype="float32")
        embeddings_index[word] = vectors

### Create Embedding Matrix

In [18]:
word_index = tokenizer.word_index
embedding_matrix = np.zeros((max_words, embedding_dim))

for word, i in word_index.items():
    if i < max_words:
        vector = embeddings_index.get(word)
        if vector is not None:
            embedding_matrix[i] = vector

### Build Model

In [19]:
model = Sequential([
    Embedding(max_words, embedding_dim, weights=[embedding_matrix],
              input_length=max_len, trainable=False),

    GRU(128),
    Dropout(0.3),

    Dense(64, activation='relu'),
    Dense(6, activation='sigmoid')
])

c:\Users\ishan\anaconda3\envs\dl\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


### Callbacks

In [20]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

lr_reduce = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=1,
    min_lr=1e-5
)

### Compile

In [22]:
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

### Train

In [23]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=128,
    callbacks=[early_stop, lr_reduce]
)

Epoch 1/10
2395/2395 ━━━━━━━━━━━━━━━━━━━━ 371s 154ms/step - accuracy: 0.9637 - loss: 0.0657 - val_accuracy: 0.9898 - val_loss: 0.0524 - learning_rate: 0.0010
Epoch 2/10
2395/2395 ━━━━━━━━━━━━━━━━━━━━ 399s 166ms/step - accuracy: 0.9770 - loss: 0.0524 - val_accuracy: 0.9919 - val_loss: 0.0491 - learning_rate: 0.0010
Epoch 3/10
2395/2395 ━━━━━━━━━━━━━━━━━━━━ 357s 149ms/step - accuracy: 0.9807 - loss: 0.0487 - val_accuracy: 0.9917 - val_loss: 0.0468 - learning_rate: 0.0010
Epoch 4/10
2395/2395 ━━━━━━━━━━━━━━━━━━━━ 343s 143ms/step - accuracy: 0.9825 - loss: 0.0460 - val_accuracy: 0.9941 - val_loss: 0.0464 - learning_rate: 0.0010
Epoch 5/10
2395/2395 ━━━━━━━━━━━━━━━━━━━━ 381s 159ms/step - accuracy: 0.9843 - loss: 0.0436 - val_accuracy: 0.9787 - val_loss: 0.0451 - learning_rate: 0.0010
Epoch 6/10
2395/2395 ━━━━━━━━━━━━━━━━━━━━ 388s 162ms/step - accuracy: 0.9835 - loss: 0.0413 - val_accuracy: 0.9818 - val_loss: 0.0437 - learning_rate: 0.0010
Epoch 7/10
2395/2395 ━━━━━━━━━━━━━━━━━━━━ 347s 145ms

### Save Model + Tokenizer

In [24]:
model.save("model.h5")

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

### Prediction Function (for testing)

In [33]:
def predict_comment(text):
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=max_len)
    pred = model.predict(pad)[0]

    labels = ['toxic','severe_toxic','obscene','threat','insult','identity_hate']
    result = dict(zip(labels, pred))

    # Add decision
    threshold = 0.5
    detected = [label for label, val in result.items() if val > threshold]

    return {
        "scores": result,
        "labels": detected if detected else ["non-toxic"]
    }

In [34]:
predict_comment("You are such a stupid idiot and nobody likes you")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step


{'scores': {'toxic': np.float32(0.9947461),
  'severe_toxic': np.float32(0.0024932877),
  'obscene': np.float32(0.7156658),
  'threat': np.float32(5.4640837e-05),
  'insult': np.float32(0.97026783),
  'identity_hate': np.float32(0.0039975657)},
 'labels': ['toxic', 'obscene', 'insult']}

In [35]:
predict_comment("What the hell is this shit? You are completely useless")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step


{'scores': {'toxic': np.float32(0.9891504),
  'severe_toxic': np.float32(0.0072849244),
  'obscene': np.float32(0.8968517),
  'threat': np.float32(0.000555727),
  'insult': np.float32(0.16940837),
  'identity_hate': np.float32(0.00017771566)},
 'labels': ['toxic', 'obscene']}

In [36]:
predict_comment("I will find you and hurt you badly")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step


{'scores': {'toxic': np.float32(0.016928302),
  'severe_toxic': np.float32(6.7831897e-06),
  'obscene': np.float32(0.0004781913),
  'threat': np.float32(0.0019934552),
  'insult': np.float32(0.0014968346),
  'identity_hate': np.float32(0.00010167655)},
 'labels': ['non-toxic']}

In [37]:
predict_comment("People like you are the worst kind of human")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step


{'scores': {'toxic': np.float32(0.46122214),
  'severe_toxic': np.float32(0.00053619256),
  'obscene': np.float32(0.011544414),
  'threat': np.float32(0.00033099297),
  'insult': np.float32(0.22687814),
  'identity_hate': np.float32(0.0094657475)},
 'labels': ['non-toxic']}

In [38]:
predict_comment("This is a great explanation, thank you for sharing")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step


{'scores': {'toxic': np.float32(1.45570775e-05),
  'severe_toxic': np.float32(8.002142e-09),
  'obscene': np.float32(2.4761625e-06),
  'threat': np.float32(1.4364356e-09),
  'insult': np.float32(6.8348254e-07),
  'identity_hate': np.float32(4.3893456e-07)},
 'labels': ['non-toxic']}

In [39]:
predict_comment("Your idea is terrible but I appreciate your effort")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step


{'scores': {'toxic': np.float32(0.012704124),
  'severe_toxic': np.float32(1.1016185e-06),
  'obscene': np.float32(8.5520194e-05),
  'threat': np.float32(1.2527439e-05),
  'insult': np.float32(0.0006667081),
  'identity_hate': np.float32(0.0001769158)},
 'labels': ['non-toxic']}

In [40]:
predict_comment("You are a complete idiot and your work is garbage")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step


{'scores': {'toxic': np.float32(0.96530676),
  'severe_toxic': np.float32(0.00027392997),
  'obscene': np.float32(0.13935903),
  'threat': np.float32(1.17127365e-05),
  'insult': np.float32(0.9031022),
  'identity_hate': np.float32(0.0014310117)},
 'labels': ['toxic', 'insult']}